# 04. 그래디언트 부스팅 구현
1. 결정트리만으로 부스팅을 직접 구현 (잔차 학습)
2. sklearn GradientBoostingRegressor와 결과 비교
3. 학습률, 트리 수, 깊이를 바꿔가며 과적합 확인

In [1]:
import numpy as np
from sklearn.tree import DecisionTreeRegressor           # 회귀 트리 (숫자를 예측하니까 Regressor)
from sklearn.model_selection import train_test_split

In [4]:
rng = np.random.default_rng(0)
X = rng.uniform(-3,3,size=(500, 1))
y = np.sin(X[:,0]) + rng.normal(scale=0.3, size=500)

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0)

In [ ]:
lr = 0.1 # 학습률
n_trees = 100 # 트리 개수

pred_tr = np.full(len(y_tr), y_tr.mean())                # 시작 예측: 전부 평균값으로 채움
pred_te = np.full(len(y_te), y_tr.mean())                # test도 train 평균으로 시작 (test 정답은 쓰면 안 됨!)
trees = [] 

for i in range(n_trees):
    residual = y_tr - pred_tr                            # 못 맞힌 만큼 = 정답 − 현재 예측

    tree = DecisionTreeRegressor(max_depth=2)            # 얕은 트리 하나
    tree.fit(X_tr, residual)                             # 정답이 아니라 잔차를 학습

    pred_tr += lr * tree.predict(X_tr)                   # 예측의 0.1배만 더하기
    pred_te += lr * tree.predict(X_te)                   # test에도 같은 트리로 똑같이 더함
    trees.append(tree)

    if (i + 1) % 10 == 0:
        mse_tr = np.mean((y_tr - pred_tr) ** 2)
        mse_te = np.mean((y_te - pred_te) ** 2)
        print(f"트리 {i+1:>3}개  train MSE={mse_tr:.4f}  test MSE={mse_te:.4f}")

트리  10개  train MSE=0.1590  test MSE=0.1697
트리  20개  train MSE=0.0886  test MSE=0.1004
트리  30개  train MSE=0.0710  test MSE=0.0922
트리  40개  train MSE=0.0650  test MSE=0.0917
트리  50개  train MSE=0.0631  test MSE=0.0922
트리  60개  train MSE=0.0614  test MSE=0.0928
트리  70개  train MSE=0.0597  test MSE=0.0945
트리  80개  train MSE=0.0580  test MSE=0.0954
트리  90개  train MSE=0.0565  test MSE=0.0966
트리 100개  train MSE=0.0552  test MSE=0.0974


In [6]:
from sklearn.ensemble import GradientBoostingRegressor

gbr = GradientBoostingRegressor(
    n_estimators=100,                                    # 트리 100개
    learning_rate=0.1,                                   # 학습률 0.1
    max_depth=2,                                         # 깊이 2
    random_state=0)
gbr.fit(X_tr, y_tr)

mse_sk = np.mean((y_te - gbr.predict(X_te)) ** 2)
print(f"직접 구현 test MSE = {np.mean((y_te - pred_te) ** 2):.4f}")
print(f"sklearn   test MSE = {mse_sk:.4f}")

직접 구현 test MSE = 0.0974
sklearn   test MSE = 0.0974
